**Define Question**

---


Pertanyaan payung:
Bagaimana kebiasaan memakai ponsel sebelum tidur berhubungan dengan kualitas tidur, sleep debt, dan kelelahan keesokan harinya?

---

Pertanyaan turunan:
Bagaimana tren sleep latency, total tidur, dan

*   kelelahan ketika durasi pakai ponsel sebelum tidur bertambah? (line chart)
*   Apakah jenis aplikasi, kecerahan layar, dan filter cahaya biru membedakan sleep latency?
*   Variabel apa yang paling berhubungan dengan kelelahan, dan kelompok mana yang paling berisiko sleep debt?


In [1]:
import pandas as pd
import numpy as np
import hashlib, os

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 150)

FILE = 'bedtime_screentime_sleep_debt_kotor.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

mentah = pd.read_csv(FILE)
print('Data mentah:', mentah.shape)

Saving bedtime_screentime_sleep_debt_kotor.csv to bedtime_screentime_sleep_debt_kotor.csv
Data mentah: (8600, 18)


In [2]:
#Mengamankan data mentah
df = mentah.copy()

jejak = []
def catat(langkah, terdampak, alasan):
    """Setiap perubahan dicatat SAAT dikerjakan, bukan belakangan."""
    jejak.append({'langkah': langkah, 'baris_terdampak': int(terdampak), 'alasan': alasan})
    print(f'  {langkah}: {int(terdampak)} baris')

print('Siap. Data mentah tetap utuh di variabel `mentah`.')

Siap. Data mentah tetap utuh di variabel `mentah`.


## **Pengecekan Quality Data**

In [7]:
# Bentuk data
display(df.head(5))
print('Jumlah baris:', len(df))
print('Jumlah kolom:', df.shape[1])

,user_id,age,gender,occupation_type,chronotype,bedtime_phone_minutes,primary_bedtime_app,screen_brightness_pct,blue_light_filter_active,caffeine_post_5pm_mg,physical_activity_min,sleep_latency_min,total_sleep_hours,deep_sleep_pct,rem_sleep_pct,morning_alarm_snoozes,next_day_fatigue_score,sleep_debt_category
0,USR-00001,29,Female,Healthcare / Shift Worker,Intermediate,179,INSTAGRAM / REDDIT,54.0,1,0.0,21,84.4,3.20,23.5,22.2,7,10.0,Severe Sleep Debt
1,USR-00002,58,Female,Student,Intermediate,163,YouTube,76.0,1,92.0,72,86.7,4.06,18.0,21.5,7,10.0,Severe Sleep Debt
2,USR-00003,41,Female,Healthcare / Shift Worker,Night Owl,100,TikTok / Reels,59.0,0,45.0,23,70.5,3.20,18.1,20.5,7,10.0,Severe Sleep Debt
3,USR-00004,36,Non-Binary,Remote Tech,Intermediate,34,YouTube,52.0,0,97.0,49,38.1,6.99,21.6,22.4,1,1.7,Mild Deficit
4,USR-00005,23,Female,Healthcare / Shift Worker,Intermediate,27,TikTok / Reels,82.0,1,0.0,38,32.3,4.58,24.1,NaN,4,5.3,Moderate Debt


Jumlah baris: 8600
Jumlah kolom: 18


In [9]:
# Periksa skema dan info data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8600 entries, 0 to 8599
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8600 non-null   object 
 1   age                       8600 non-null   object 
 2   gender                    8600 non-null   object 
 3   occupation_type           8429 non-null   object 
 4   chronotype                8473 non-null   object 
 5   bedtime_phone_minutes     8600 non-null   object 
 6   primary_bedtime_app       8600 non-null   object 
 7   screen_brightness_pct     8429 non-null   float64
 8   blue_light_filter_active  8600 non-null   object 
 9   caffeine_post_5pm_mg      8429 non-null   float64
 10  physical_activity_min     8600 non-null   int64  
 11  sleep_latency_min         8344 non-null   float64
 12  total_sleep_hours         8600 non-null   float64
 13  deep_sleep_pct            8343 non-null   float64
 14  rem_slee

Dari pemeriksaan tipe data, `age`, `bedtime_phone_minutes`, dan
`rem_sleep_pct` terbaca sebagai teks, padahal nilainya seharusnya numerik.

Kemungkinan penyebabnya adalah satuan `tahun`/`min` yang tercampur dalam
nilai, serta tanda `-` pada `rem_sleep_pct`. Nama tipe yang tampil dapat
berbeda antarversi pandas (`object` atau `str`), sehingga perlu diperiksa
isinya sebelum konversi.

In [13]:
# cek spasi berlebih awal/akhir
laporan_spasi = []

for kol in df.columns:
    tipe_teks = (
        pd.api.types.is_string_dtype(df[kol])
        or pd.api.types.is_object_dtype(df[kol])
    )

    if tipe_teks:
        s = df[kol].astype('string')
        jumlah = int(
            s.ne(s.str.strip()).fillna(False).sum()
        )

        if jumlah:
            laporan_spasi.append({
                'kolom': kol,
                'sel_dengan_spasi_tepi': jumlah
            })

display(pd.DataFrame(laporan_spasi))

,kolom,sel_dengan_spasi_tepi
0,user_id,20
1,gender,120
2,occupation_type,179
3,chronotype,74
4,primary_bedtime_app,107


In [14]:
# Melakukan pengecekan variasi kategori dan ejaan
kolom_kat = [
    'gender',
    'occupation_type',
    'chronotype',
    'primary_bedtime_app',
    'blue_light_filter_active',
    'sleep_debt_category'
]

def kunci(s):
    return (
        s.astype('string')
         .str.strip()
         .str.lower()
         .str.replace(r'[\s_-]+', ' ', regex=True)
    )

alias = {
    'gender': {
        'f': 'female',
        'm': 'male'
    },
    'blue_light_filter_active': {
        'yes': '1',
        'true': '1',
        'no': '0',
        'false': '0'
    },
    'sleep_debt_category': {
        'moderat debt': 'moderate debt',
        'mild defisit': 'mild deficit',
        'optimal recovry': 'optimal recovery',
        'sever sleep debt': 'severe sleep debt'
    }
}

for kol in kolom_kat:
    normal = kunci(df[kol])
    semantik = normal.replace(alias.get(kol, {}))

    print(f'\n=== {kol} ===')
    print('Nilai unik mentah:', df[kol].nunique(dropna=True))
    print('Setelah format diseragamkan:', normal.nunique(dropna=True))
    print('Setelah alias/ejaan disatukan:', semantik.nunique(dropna=True))
    print(
        'Baris dengan alias/ejaan berbeda:',
        int(normal.ne(semantik).fillna(False).sum())
    )

    varian = (
        df[[kol]]
        .assign(_kunci=normal)
        .dropna()
        .groupby('_kunci')[kol]
        .agg(lambda x: sorted(x.unique().tolist()))
    )

    display(varian.rename('varian_mentah').to_frame())


=== gender ===
Nilai unik mentah: 14
Setelah format diseragamkan: 5
Setelah alias/ejaan disatukan: 3
Baris dengan alias/ejaan berbeda: 136


,varian_mentah
_kunci,
f,[F]
female,"[FEMALE, Female, Female , female]"
m,[M]
male,"[ Male, MALE, Male, male]"
non binary,"[NON-BINARY, Non Binary, Non-Binary, non-binary]"



=== occupation_type ===
Nilai unik mentah: 25
Setelah format diseragamkan: 5
Setelah alias/ejaan disatukan: 5
Baris dengan alias/ejaan berbeda: 0


,varian_mentah
_kunci,
corporate 9 to 5,"[ Corporate 9-to-5, CORPORATE 9-TO-5, Corporat..."
freelance / creative,"[ Freelance / Creative, FREELANCE / CREATIVE, ..."
healthcare / shift worker,"[ Healthcare / Shift Worker, HEALTHCARE / SHIF..."
remote tech,"[ Remote Tech, REMOTE TECH, Remote Tech, Remot..."
student,"[ Student, STUDENT, Student, Student , student]"



=== chronotype ===
Nilai unik mentah: 11
Setelah format diseragamkan: 3
Setelah alias/ejaan disatukan: 3
Baris dengan alias/ejaan berbeda: 0


,varian_mentah
_kunci,
intermediate,"[Intermediate, Intermediate , intermediate]"
morning lark,"[Morning Lark, Morning Lark , Morning_Lark, mo..."
night owl,"[Night Owl, Night Owl , Night_Owl, night owl]"


... (2 outputs left out)


=== blue_light_filter_active ===
Nilai unik mentah: 8
Setelah format diseragamkan: 6
Setelah alias/ejaan disatukan: 2
Baris dengan alias/ejaan berbeda: 688


,varian_mentah
_kunci,
0,[0]
1,[1]
false,[FALSE]
no,"[No, no]"
true,[TRUE]
yes,"[Yes, yes]"



=== sleep_debt_category ===
Nilai unik mentah: 8
Setelah format diseragamkan: 8
Setelah alias/ejaan disatukan: 4
Baris dengan alias/ejaan berbeda: 40


,varian_mentah
_kunci,
mild deficit,[Mild Deficit]
mild defisit,[Mild Defisit]
moderat debt,[Moderat Debt]
moderate debt,[Moderate Debt]
optimal recovery,[Optimal Recovery]
optimal recovry,[Optimal Recovry]
sever sleep debt,[Sever Sleep Debt]
severe sleep debt,[Severe Sleep Debt]


In [15]:
# Melihat missing value termasuk tanda '-'
penanda = pd.DataFrame(index=df.index)

for kol in df.columns:
    teks = df[kol].astype('string').str.strip().str.lower()
    penanda[kol] = teks.isin([
        '', '-', 'na', 'n/a', 'null', 'none'
    ])

hilang = df.isna() | penanda

kelengkapan = pd.DataFrame({
    'NaN': df.isna().sum(),
    'penanda_kosong': penanda.sum(),
    'total': hilang.sum()
})

kelengkapan['persen'] = (
    100 * kelengkapan['total'] / len(df)
).round(2)

display(
    kelengkapan[kelengkapan['total'].gt(0)]
    .sort_values('total', ascending=False)
)

print('Total sel tidak tersedia:', int(hilang.sum().sum()))
print(
    'Baris dengan ≥1 nilai tidak tersedia:',
    int(hilang.any(axis=1).sum())
)
print(
    'Baris dengan ≥2 nilai tidak tersedia:',
    int(hilang.sum(axis=1).ge(2).sum())
)

kolom_hilang = kelengkapan.index[
    kelengkapan['total'].gt(0)
].tolist()

print('\nPola nilai hilang terbanyak:')
display(
    hilang.loc[hilang.any(axis=1), kolom_hilang]
    .value_counts()
    .head(8)
    .rename('jumlah_baris')
    .reset_index()
)

,NaN,penanda_kosong,total,persen
deep_sleep_pct,257,0,257,2.99
sleep_latency_min,256,0,256,2.98
occupation_type,171,0,171,1.99
caffeine_post_5pm_mg,171,0,171,1.99
screen_brightness_pct,171,0,171,1.99
chronotype,127,0,127,1.48
rem_sleep_pct,36,25,61,0.71


Total sel tidak tersedia: 1214
Baris dengan ≥1 nilai tidak tersedia: 1140
Baris dengan ≥2 nilai tidak tersedia: 71

Pola nilai hilang terbanyak:


,occupation_type,chronotype,screen_brightness_pct,caffeine_post_5pm_mg,sleep_latency_min,deep_sleep_pct,rem_sleep_pct,jumlah_baris
0,False,False,False,False,False,True,False,231
1,False,False,False,False,True,False,False,227
2,False,False,True,False,False,False,False,154
3,True,False,False,False,False,False,False,152
4,False,False,False,True,False,False,False,145
5,False,True,False,False,False,False,False,112
6,False,False,False,False,False,False,True,48
7,False,False,False,True,True,False,False,11


In [18]:
# Apakah angka dapat dibaca?
kolom_angka = [
    'age',
    'bedtime_phone_minutes',
    'screen_brightness_pct',
    'caffeine_post_5pm_mg',
    'physical_activity_min',
    'sleep_latency_min',
    'total_sleep_hours',
    'deep_sleep_pct',
    'rem_sleep_pct',
    'morning_alarm_snoozes',
    'next_day_fatigue_score'
]

sufiks = {
    'age': r'(?i)\s*tahun\s*$',
    'bedtime_phone_minutes': r'(?i)\s*min\s*$'
}

angka = pd.DataFrame(index=df.index)
format_angka = []

for kol in kolom_angka:
    asli = df[kol].astype('string').str.strip()

    if kol in sufiks:
        siap = asli.str.replace(
            sufiks[kol], '', regex=True
        )
    else:
        siap = asli

    angka[kol] = pd.to_numeric(siap, errors='coerce')

    bersatuan = asli.notna() & asli.ne(siap)
    gagal = (
        asli.notna()
        & ~hilang[kol]
        & angka[kol].isna()
    )

    format_angka.append({
        'kolom': kol,
        'dtype_awal': str(df[kol].dtype),
        'nilai_dengan_satuan': int(bersatuan.sum()),
        'gagal_dibaca': int(gagal.sum()),
        'contoh_gagal': (
            asli[gagal]
            .drop_duplicates()
            .head(5)
            .tolist()
        )
    })

display(pd.DataFrame(format_angka).set_index('kolom'))

,dtype_awal,nilai_dengan_satuan,gagal_dibaca,contoh_gagal
kolom,,,,
age,object,31,0,[]
bedtime_phone_minutes,object,51,0,[]
screen_brightness_pct,float64,0,0,[]
caffeine_post_5pm_mg,float64,0,0,[]
physical_activity_min,int64,0,0,[]
sleep_latency_min,float64,0,0,[]
total_sleep_hours,float64,0,0,[]
deep_sleep_pct,float64,0,0,[]
rem_sleep_pct,object,0,0,[]


In [19]:
# Pengecekan Format Id dan duplikasi
id_bersih = df['user_id'].astype('string').str.strip()

id_tidak_sesuai = ~id_bersih.str.fullmatch(
    r'USR-\d{5}',
    na=False
)

print(
    'ID tidak sesuai pola USR-xxxxx:',
    int(id_tidak_sesuai.sum())
)
print(
    'ID berulang setelah spasi tepi diabaikan:',
    int(id_bersih.duplicated().sum())
)
print(
    'Baris duplikat persis:',
    int(df.duplicated().sum())
)

baris_id_ulang = (
    df.assign(_id=id_bersih)
      .loc[id_bersih.duplicated(keep=False)]
)

kolom_selain_id = df.columns.drop('user_id').tolist()

id_isi_berbeda = (
    baris_id_ulang.groupby('_id')[kolom_selain_id]
    .nunique(dropna=False)
    .gt(1)
    .any(axis=1)
    .sum()
)

print(
    'ID berulang dengan isi berbeda:',
    int(id_isi_berbeda)
)

ID tidak sesuai pola USR-xxxxx: 0
ID berulang setelah spasi tepi diabaikan: 100
Baris duplikat persis: 100
ID berulang dengan isi berbeda: 0


In [20]:
# Cek nilai di luar nilai rentang dasar
batas = {
    'age': (0, 120),
    'bedtime_phone_minutes': (0, 1440),
    'screen_brightness_pct': (0, 100),
    'caffeine_post_5pm_mg': (0, None),
    'physical_activity_min': (0, 1440),
    'sleep_latency_min': (0, 1440),
    'total_sleep_hours': (0, 24),
    'deep_sleep_pct': (0, 100),
    'rem_sleep_pct': (0, 100),
    'morning_alarm_snoozes': (0, None),
    'next_day_fatigue_score': (1, 10)
}

pelanggaran = pd.DataFrame(index=df.index)
laporan_batas = []

for kol, (bawah, atas) in batas.items():
    buruk = angka[kol].lt(bawah)

    if atas is not None:
        buruk = buruk | angka[kol].gt(atas)

    buruk = buruk.fillna(False)
    pelanggaran[kol] = buruk

    if buruk.any():
        laporan_batas.append({
            'kolom': kol,
            'batas_bawah': bawah,
            'batas_atas': atas,
            'jumlah': int(buruk.sum()),
            'contoh_nilai': (
                df.loc[buruk, kol]
                .value_counts()
                .head(5)
                .to_dict()
            )
        })

display(pd.DataFrame(laporan_batas))

print(
    'Baris dengan ≥1 pelanggaran batas:',
    int(pelanggaran.any(axis=1).sum())
)

,kolom,batas_bawah,batas_atas,jumlah,contoh_nilai
0,age,0,120.0,6,"{'-5': 4, '999': 2}"
1,screen_brightness_pct,0,100.0,6,"{130.0: 3, -20.0: 2, 150.0: 1}"
2,caffeine_post_5pm_mg,0,NaN,2,{-50.0: 2}
3,sleep_latency_min,0,1440.0,2,{-10.0: 2}
4,total_sleep_hours,0,24.0,6,"{-3.0: 3, 25.0: 3}"


Baris dengan ≥1 pelanggaran batas: 22


In [21]:
# Cek konsistensi kolom
persen_fase = (
    angka['deep_sleep_pct']
    + angka['rem_sleep_pct']
)

fase_lebih_100 = persen_fase.gt(100).fillna(False)

tidur_nol_berfase = (
    angka['total_sleep_hours'].eq(0)
    & (
        angka['deep_sleep_pct'].gt(0)
        | angka['rem_sleep_pct'].gt(0)
    )
).fillna(False)

print(
    'Deep sleep + REM > 100%:',
    int(fase_lebih_100.sum())
)
print(
    'Tidur 0 jam tetapi fase tidur positif:',
    int(tidur_nol_berfase.sum())
)

display(
    df.loc[
        tidur_nol_berfase,
        [
            'user_id',
            'total_sleep_hours',
            'deep_sleep_pct',
            'rem_sleep_pct',
            'sleep_debt_category'
        ]
    ]
)

Deep sleep + REM > 100%: 0
Tidur 0 jam tetapi fase tidur positif: 2


,user_id,total_sleep_hours,deep_sleep_pct,rem_sleep_pct,sleep_debt_category
2361,USR-02336,0.0,19.9,21.6,Moderate Debt
8428,USR-08331,0.0,22.5,18.8,Optimal Recovery


In [22]:
# Cek Kandidat nilai ekstrem dengan IQR
laporan_ekstrem = []

for kol in kolom_angka:
    valid = angka[kol].mask(pelanggaran[kol]).dropna()
    q1, q3 = valid.quantile([0.25, 0.75])
    iqr = q3 - q1

    if iqr == 0:
        continue

    batas_bawah = q1 - 1.5 * iqr
    batas_atas = q3 + 1.5 * iqr

    kandidat = (
        (
            angka[kol].lt(batas_bawah)
            | angka[kol].gt(batas_atas)
        )
        & ~pelanggaran[kol]
    ).fillna(False)

    if kandidat.any():
        laporan_ekstrem.append({
            'kolom': kol,
            'batas_IQR_bawah': round(batas_bawah, 2),
            'batas_IQR_atas': round(batas_atas, 2),
            'jumlah': int(kandidat.sum()),
            'min_kandidat': angka.loc[kandidat, kol].min(),
            'max_kandidat': angka.loc[kandidat, kol].max()
        })

display(pd.DataFrame(laporan_ekstrem))
print('Outlier IQR perlu ditinjau; tidak otomatis salah.')

,kolom,batas_IQR_bawah,batas_IQR_atas,jumlah,min_kandidat,max_kandidat
0,bedtime_phone_minutes,-41.0,151.0,237,152.0,180.0
1,caffeine_post_5pm_mg,-82.5,137.5,481,138.0,2500.0
2,physical_activity_min,-22.5,93.5,40,94.0,112.0
3,sleep_latency_min,-3.9,81.7,242,81.8,999.0
4,total_sleep_hours,2.1,10.5,2,0.0,0.0
5,deep_sleep_pct,13.4,30.2,49,8.1,13.3
6,rem_sleep_pct,12.3,25.9,76,9.6,27.0


Outlier IQR perlu ditinjau; tidak otomatis salah.


### Kesimpulan diagnosis kualitas data

Dataset berisi 8.600 baris dan 18 kolom. Ditemukan 1.214 sel yang
tidak tersedia pada 1.140 baris: 1.189 nilai NaN dan 25 tanda `-`
pada `rem_sleep_pct`.

Terdapat 100 baris duplikat persis dan 100 ID berulang, tanpa ID
berulang yang memiliki isi berbeda. Sebanyak 31 nilai `age` memuat
satuan `tahun`, dan 51 nilai `bedtime_phone_minutes` memuat `min`.
Kategori juga memiliki variasi huruf, spasi, alias, serta 40 salah
eja pada `sleep_debt_category`.

Sebanyak 22 baris melanggar batas nilai dasar. Selain itu, 2 baris
mencatat durasi tidur 0 jam tetapi memiliki persentase fase tidur
positif. Tidak ditemukan jumlah deep sleep dan REM yang melebihi 100%.

Hasil IQR adalah kandidat nilai ekstrem untuk ditinjau, bukan
otomatis data salah. Batas yang digunakan merupakan aturan
pemeriksaan awal berdasarkan satuan dan skala kolom. Ketepatan
`sleep_debt_category` per baris belum dapat diuji tanpa definisi
operasional pembentukan label.